Notebook realizado por Juan David Sánchez. Contacto: juan.sanchez34@udea.edu.co

# **Sesión 5 - Similitud química**

El dataset 1 contiene información acerca de las moléculas con bioactividad reportada frente al target Proteína Precursora Amiloidea (APP), el cual fue obtenido a partir de la base de datos PubChem.

El dataset 2 y el dataset 3 contienen información referente moléculas con bioactividad hacia los targets GPR40 y G9a. Son bases de datos curada por DIFACQUIM.

In [2]:
# Importing necessary libraries

import os
import sys
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import urllib.request

import pip
! pip install rdkit
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

📗📘📙📌 **Importe y homogenización de las bases de datos** 📌📙📘📗

In [34]:
# Importing databases
APP_df = pd.read_csv("https://raw.githubusercontent.com/Sanchez-Juan/Course__Chemoinformatics-applied-to-drug-design/refs/heads/main/app_pubchem.csv") # APP dataset from PubChem
GPR40_df = pd.read_csv("https://raw.githubusercontent.com/DIFACQUIM/Cursos/main/Datasets/08_Similitud_GPR40.csv") # GPR40 dataset
G9a_df = pd.read_csv("https://raw.githubusercontent.com/DIFACQUIM/Cursos/refs/heads/main/Datasets/08_Similitud_compounds_g9a.csv") # G9a dataset

print("🔶 Las columnas que contiene cada base de datos son:")
print(f"- APP dataset: {APP_df.columns.tolist()}")
print(f"- GPR40 dataset: {GPR40_df.columns.tolist()}")
print(f"- G9a dataset: {G9a_df.columns.tolist()}")
print(f"\n🔷 El número de compuestos en cada base de datos es: APP-PubChem: {APP_df.shape[0]}, GPR40: {GPR40_df.shape[0]}, G9a: {G9a_df.shape[0]}")

🔶 Las columnas que contiene cada base de datos son:
- APP dataset: ['Molecule_id', 'Activity Value [nM]', 'SMILES', 'Molecules', 'MW', 'HBA', 'HBD', 'logP', 'TPSA', 'CSP3', 'NumRings', 'HetAtoms', 'RotBonds', 'Murcko_SMILES', 'Scaffold', 'Source']
- GPR40 dataset: ['Molecule ChEMBL ID', 'Molecule Name', 'Molecule Max Phase', 'Molecular Weight', '#RO5 Violations', 'AlogP', 'Compound Key', 'Smiles', 'Standard Type', 'Standard Relation', 'Standard Value', 'Standard Units', 'pChEMBL Value', 'Data Validity Comment', 'Comment', 'Uo Units', 'Ligand Efficiency BEI', 'Ligand Efficiency LE', 'Ligand Efficiency LLE', 'Ligand Efficiency SEI', 'Potential Duplicate', 'Assay ChEMBL ID', 'Assay Description', 'Assay Type', 'BAO Format ID', 'BAO Label', 'Assay Organism', 'Assay Tissue ChEMBL ID', 'Assay Tissue Name', 'Assay Cell Type', 'Assay Subcellular Fraction', 'Assay Parameters', 'Assay Variant Accession', 'Assay Variant Mutation', 'Target ChEMBL ID', 'Target Name', 'Target Organism', 'Target Type'

In [35]:
# Renaming and selecting relevant columns in order to concatenate the databases

APP_df = APP_df[["Source", "Molecule_id", "Activity Value [nM]", "SMILES"]]
APP_df = APP_df.rename(columns={"Source": "Dataset"})

GPR40_df["Dataset"] = "CheMBL"
GPR40_df = GPR40_df[["Dataset", "Molecule ChEMBL ID", "Standard Value", "Smiles"]]
GPR40_df = GPR40_df.rename(columns={"Molecule ChEMBL ID": "Molecule_id", "Standard Value": "Activity Value [nM]", "Smiles": "SMILES"})

G9a_df["Dataset"] = "CheMBL"
G9a_df = G9a_df[["Dataset", "molecule_chembl_id", "IC50", "smiles"]]
G9a_df = G9a_df.rename(columns={"molecule_chembl_id": "Molecule_id", "IC50": "Activity Value [nM]", "smiles": "SMILES"})

# Concatenating the three databases into a single dataframe
data = pd.concat([APP_df, GPR40_df, G9a_df], ignore_index=True)
print(f"🔷 El número total de compuestos en la base de datos combinada es: {data.shape[0]}")
data.sample(5)

🔷 El número total de compuestos en la base de datos combinada es: 2018


,Dataset,Molecule_id,Activity Value [nM],SMILES
881,PubChem,155514094,3630.0,CN(CCCNC(=O)C1=CC(=C(C=C1O)C(=O)NCCCN(C)CC2=CC...
809,PubChem,70691586,62700.0,CCCCCCCCCCCCCCC[C@H](C(=O)N[C@@H](CO[C@H]1[C@@...
1035,PubChem,164627131,9270.0,CC1=CC2=C(C=C1NCC3=C(C(=CC=C3)Br)O)C(=O)N4CCCC...
512,PubChem,102115496,15900.0,COC1=CC=C(C=C1)C2=CC(=O)C3=C(O2)C(=C(C=C3O)O)C...
1970,CheMBL,CHEMBL3109629,14400,COc1cc2cc(N)[nH]c2cc1OC


📗📘📙📌 **X** 📌📙📘📗

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys, rdFingerprintGenerator
from rdkit.Chem.Pharm2D import Generate, Gobbi_Pharm2D

def calcular_fingerprints_avanzados(df, mol_col='ROMol_H'):
    """
    Calcula múltiples tipos de huellas moleculares (2D, 3D, ECFP6, MACCS)
    y las añade como nuevas columnas al DataFrame.
    """
    # Generadores
    ecfp6_gen = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=1024)
    factory = Gobbi_Pharm2D.factory
    ecfp6_list, maccs_list, fp2d_list, fp3d_list = [], [], [], []
    time_0 = time.time()

    for m in df[mol_col]:
        if m is None:
            for l in [ecfp6_list, maccs_list, fp2d_list, fp3d_list]: l.append(None)
            continue

        # --- ECFP6 y MACCS ---
        ecfp6_list.append(ecfp6_gen.GetFingerprint(m))
        maccs_list.append(MACCSkeys.GenMACCSKeys(m))

        # --- Farmacóforo 2D ---
        fp2d_list.append(Generate.Gen2DFingerprint(m, factory))

        # --- Farmacóforo 3D ---
        # Intentar generar conformador 3D (necesario para matriz de distancia 3D)
        try:
            m_3d = Chem.Mol(m)
            m_3d = Chem.AddHs(m_3d) # Los hidrógenos son clave para 3D
            res = AllChem.EmbedMolecule(m_3d, AllChem.ETKDG())

            if res != -1: # Si la optimización tuvo éxito
                dist_mat = Chem.Get3DDistanceMatrix(m_3d)
                fp3d = Generate.Gen2DFingerprint(m_3d, factory, dMat=dist_mat)
                fp3d_list.append(fp3d)
            else:
                fp3d_list.append(None)
        except:
            fp3d_list.append(None)

    # Asignar listas al DataFrame de una sola vez
    df['ECFP6'] = ecfp6_list
    df['MACCS'] = maccs_list
    df['Fp2D'] = fp2d_list
    df['Fp3D'] = fp3d_list
    time_1 = time.time()
    print(f"Tiempo de ejecución: {time_1 - time_0} segundos")
    display(df)

    return df

📗📘📙📌 **X** 📌📙📘📗

📗📘📙📌 **X** 📌📙📘📗